# Suite STO — Stores

A store is the root object of a body of data: it holds schemas by name and the objects built with them, and everything
that looks a schema up takes one. `Proxies.OfStore` and `Bindings.OfStore` implement the protocol `Stores.Store`.
Stores are isolated from one another; objects move between them as snapshots.

In [ ]:
from mbse.Schemas.Framework import Bindings as Bd, JSON, Modules, Proxies, Schemas as S, Stores, Validators
from support import conformance_problems, instance_problems, raises, text

Item = S.OfObject.Builder().ref().properties(text("name")).create()
Holds = S.OfRelation.Builder().links("owner", "item").properties(text("label")).create()
Box = S.OfObject.Builder().ref().properties(text("name")).relations(lambda r: r.name("items").of(Holds).me("owner")).create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("boxes").of(Holds).me("item")).update()
Port = S.OfObject.Builder().properties(text("name")).relations(lambda r: r.name("wired").of(Holds).me("owner")).create()
Card = S.OfObject.Builder().ref().properties(lambda p: p.name("port").of(Port)).create()


def stocked(*names):
    """A proxy store with the schemas named, of Item, Box, Holds and Card."""
    store = Proxies.OfStore()
    for name in names:
        store.register(name, {"Item": Item, "Box": Box, "Holds": Holds, "Card": Card}[name])
    return store

## STO-01 · A store holds schemas by name

In [ ]:
store = stocked("Item", "Box", "Holds")
assert store.names() == ("Schemas.Module", "Item", "Box", "Holds")  # every proxy store starts with the meta-schemas
assert store.schema("Box") is Box and store.registered("Holds") is Holds and store.name_of(Item) == "Item"
with raises(ValueError, match="schema 'Box' is already registered"):
    store.register("Box", Box)
with raises(AttributeError, match="no schema registered as 'Nope'"):
    store.schema("Nope")
with raises(TypeError, match="'Holds' is a relation; no relation builder is exposed"):
    store.schema("Holds")
with raises(LookupError, match="no schema registered as 'Nope'"):
    store.registered("Nope")
with raises(LookupError, match="schema is not registered"):
    store.name_of(Port)
store.register("register", Item)  # a name a method shadows is reached through builder()
assert store.builder("register").name("r").create().schema_name() == "register"
with raises(AttributeError, match="_hidden"):
    getattr(store, "_hidden")
assert store.Box().name("b").create().name == "b" and store.builder("Box").name("c").create().name == "c"

## STO-02 · A store holds the objects built through it

In [ ]:
store = stocked("Item", "Box", "Holds", "Card")
assert store.extent("Box") == () and store.extent("Card") == ()
a = store.Item().name("a").create()
box = store.Box().name("box").items(lambda e: e.item(a).label("first")).create()
box2 = store.Box(box).name("box2").clone()
assert store.Box(box).name("renamed").update() is box  # an update makes nothing new
store.Box(box).items(lambda e: e.item(lambda i: i.name("inline"))).update()  # an object created through a link
assert store.extent("Box") == (box, box2)
assert [i.name for i in store.extent("Item")] == ["a", "inline"]
card = store.Card().port(lambda p: p.name("p1")).create()
assert store.extent("Card") == (card,)  # its value object is reached through it, not an extent
loaded = JSON.FromJSON(store).Reachable(Box, JSON.ToJSON(store).Reachable(Box, box))  # decoding builds in the store
# The snapshot holds box2 too, which shares an item with box; the root is built first.
assert [b.name for b in store.extent("Box")] == ["renamed", "box2", "renamed", "box2"] and store.extent("Box")[2] is loaded
assert len(store.extent("Item")) == 4
with raises(AttributeError, match="no schema registered as 'Nope'"):
    store.extent("Nope")
with raises(TypeError, match="'Holds' is a relation"):
    store.extent("Holds")

## STO-03 · Stores are isolated; objects move between them as snapshots

In [ ]:
one, two = stocked("Item", "Box", "Holds"), stocked("Item", "Box", "Holds")
Other = S.OfObject.Builder().ref().properties(text("title")).create()
three = Proxies.OfStore()
three.register("Box", Other)  # the same name, another schema
assert three.Box().title("t").create().title == "t"
x = one.Item().name("x").create()
y = two.Item().name("y").create()
with raises(TypeError, match="the object belongs to another store"):
    one.Box().items(lambda e: e.item(y))
with raises(TypeError, match="the object belongs to another store"):
    two.Item(x)
b = one.Box().name("b").items(lambda e: e.item(x).label("l")).create()
assert Proxies.store_of(b) is one and Proxies.store_of(b) is not two
moved = JSON.FromJSON(two).Reachable(Box, JSON.ToJSON(one).Reachable(Box, b))
assert Proxies.store_of(moved) is two and moved is not b and [i.name for i in two.extent("Item")] == ["y", "x"]
assert Validators.Validate(two).Reachable(Box, moved) == []
assert Validators.Validate(three)(Box, b) == ["the value is a 'Box', not an instance of the given schema"]
card = stocked("Card").Card().port(lambda p: p.name("p")).create()
assert Proxies.store_of(card.port) is Proxies.store_of(card)  # a value object's store is its owner's
with raises(TypeError, match="the value does not belong to a store"):
    Proxies.store_of("text")
module = Modules.module(one, {"Item": Item})
assert Proxies.store_of(module) is one and one.extent("Schemas.Module") == (module,)

## STO-04 · Every implementation meets the protocol

In [ ]:
assert conformance_problems(Proxies.OfStore, Stores.Store) == []
assert conformance_problems(Bd.OfStore, Stores.Store) == []
assert instance_problems(Proxies.OfStore(), Stores.Store) == []


class Thing:
    def __init__(self, name=None):
        self.name = name

    def identity(self):
        return id(self)

    def schema_name(self):
        return "Item"

    def owner(self):
        return None

    def accept(self, visitor):
        Bd.accept(THING, self, visitor)


THING = Bd.Binding(Item, lambda t: Bd.State({"name": t.name}), lambda s: Thing(s.values.get("name")),
                   lambda t, s: (setattr(t, "name", s.values.get("name")), t)[1], implied=["boxes"])
bound = Bd.OfStore({"Item": (Item, lambda instance=None: Bd.Builder(THING, instance))}, {"Holds": Holds})
assert instance_problems(bound, Stores.Store) == []
assert bound.names() == ("Item", "Holds")  # a store of bound classes holds no meta-schemas
first = bound.Item().property("name", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("first")))).create()
copy = bound.builder("Item", first).clone()
assert bound.builder("Item", first).update() is first
Bd.Builder(THING).create()  # a builder the store did not make adds nothing
assert bound.extent("Item") == (first, copy) and bound.member(first, "name") == "first"
assert JSON.FromJSON(bound)(Item, JSON.ToJSON(bound)(Item, first)).name == "first" and len(bound.extent("Item")) == 3
with raises(AttributeError, match="Nope"):
    bound.Nope
with raises(TypeError, match="'Holds' is a relation"):
    bound.builder("Holds")
with raises(TypeError, match="'Holds' is a relation"):
    bound.extent("Holds")

## STO-05 · A value object not yet placed belongs to no store

In [ ]:
loose = Proxies._RecordBuilder(Port).name("loose").build()  # built, but not placed in an owner
seen = []
loose.accept(type("Recorder", (), {"identify": lambda self, v: None,
                                   "property": lambda self, n, cb: seen.append(n),
                                   "adjacency": lambda self, n, cb: seen.append(n)})())
assert seen == ["name"]  # its properties, and no entries
with raises(TypeError, match="the value does not belong to a store"):
    Proxies.store_of(loose)
with raises(TypeError, match="an object is created through a link only by a store's builder"):
    Proxies._RecordBuilder(Port).wired(lambda e: e.item(lambda i: i.name("new")))